In [1]:
import pandas as pd
from pandas.api.types import (
    is_datetime64_any_dtype,
    is_numeric_dtype,
)
from pandas.core.dtypes.dtypes import DatetimeTZDtype
import numpy as np
import warnings
from datetime import datetime, timedelta
import os
from pathlib import Path
import re
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
# Change working directory to project root
PROJECT_ROOT = Path().absolute().parent if Path().absolute().name == 'processors' else Path().absolute()
os.chdir(PROJECT_ROOT)
!pwd

/Users/phatvu/Documents/Dev-Drive-Local/crypto-price-forecaster-glm


In [3]:
# Import project configuration
import sys
sys.path.append('.')
from config import *

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

hf_key = os.getenv("hf_key")
if hf_key is None:
    print("Warning: hf_key not found in environment variables, please set it in the .env file.")
else:
    print("hf_key loaded from environment variables:", hf_key.replace(hf_key[3:-2], "****"))

hf_key loaded from environment variables: hf_****ak


## Utils

### Convert Timestamp

In [5]:
def convert_timestamp(
    df: pd.DataFrame,
    col: str,
    tz: str = "UTC",
    round_to: str = "s"   # nearest second (lowercase = non-deprecated)
) -> pd.DataFrame:
    """Convert and normalize a timestamp column to UTC, with optional rounding.
       Includes preprocessing for mixed ISO 8601 strings (with or without microseconds).
    """

    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found.")

    df_copy = df.copy()
    s = df_copy[col]
    dtype = s.dtype

    # datetime-like, tz-aware or tz-naive
    if is_datetime64_any_dtype(s) or isinstance(dtype, DatetimeTZDtype):
        ts = pd.to_datetime(s, utc=True).dt.tz_convert(tz)

    # numeric epoch
    elif is_numeric_dtype(s):
        s_nonnull = s.dropna()
        if s_nonnull.empty:
            raise ValueError("Timestamp column is empty.")

        max_val = s_nonnull.max()
        # Determine unit based on magnitude
        if max_val > 1e17:
            unit = "ns"
        elif max_val > 1e14:
            unit = "us"
        elif max_val > 1e11:
            unit = "ms"
        else:
            unit = "s"

        ts = pd.to_datetime(s, unit=unit, utc=True).dt.tz_convert(tz)

    # string / mixed formats (including the problematic microsecond format)
    else:
        # --- FIX LỖI NaT: Chuẩn hóa chuỗi trước khi parse ---
        # 1. Chuyển đổi sang chuỗi để sử dụng .str
        s_str = s.astype(str)
        
        # 2. Loại bỏ microsecond để tránh lỗi khi Pandas đoán format
        # Pattern: tìm .[số]...[số]+00:00 (ví dụ: .844001+00:00) và thay bằng +00:00
        # Đây là bước quan trọng để thống nhất định dạng
        s_clean = s_str.str.replace(r'\.\d{3,6}\+00:00', r'+00:00', regex=True)
        
        # 3. Chuyển đổi sang datetime
        ts = pd.to_datetime(s_clean, utc=True, errors="coerce")
        
        if ts.isna().all():
            raise ValueError(f"Cannot parse timestamps in column '{col}'.")
        ts = ts.dt.tz_convert(tz)

    # rounding (use lowercase 's', 'ms', 'us', etc.)
    if round_to is not None:
        ts = ts.dt.round(round_to)

    df_copy[col] = ts
    return df_copy

### Crawl News

In [6]:
import os
import re
from bs4 import BeautifulSoup, NavigableString
from typing import Dict, List, Union

def extract_article(source: str) -> dict:
    """
    Extracts structured content from Bitcoin Magazine HTML.
    Optimized for 2012-2025 layouts with proper paragraph separation.
    """
    # 1. Load content
    html_content = ""
    if os.path.exists(source) and os.path.isfile(source):
        try:
            with open(source, "r", encoding="utf-8") as f:
                html_content = f.read()
        except Exception:
            return {}
    else:
        html_content = source

    soup = BeautifulSoup(html_content, 'html.parser')
    data = {}

    # 2. Metadata extraction
    meta_title = soup.find("meta", property="og:title")
    data['title'] = meta_title.get("content", "").strip() if meta_title else ""
    if not data['title']:
        t_tag = soup.find("title")
        data['title'] = t_tag.get_text(strip=True) if t_tag else "Unknown"

    meta_auth = soup.find("meta", attrs={"name": "author"})
    data['author'] = meta_auth.get("content", "").strip() if meta_auth else "Unknown"

    meta_date = soup.find("meta", property="article:published_time")
    data['date'] = meta_date.get("content", "").strip() if meta_date else None

    data['tags'] = [
        t.get("content").strip() 
        for t in soup.find_all("meta", property="article:tag") 
        if t.get("content")
    ]

    # 3. Locate main content container
    # Priority ordered selectors for various site versions
    selectors = [
        ".tdb_single_content .tdb-block-inner",
        ".tdb_single_content",
        ".entry-content",
        ".post-content",
        "article",
        ".tdb-block-inner",
        ".main-content"
    ]

    container = None
    for sel in selectors:
        container = soup.select_one(sel)
        if container:
            break

    if not container:
        data['content'] = ""
        return data

    # 4. Remove technical and UI junk
    junk_tags = [
        "script", "style", "iframe", "noscript", "svg", "form", "button", 
        "input", ".td-social-sharing-buttons", ".share", ".navigation", 
        ".menu", ".sidebar", ".comments-area", ".related-posts", ".widget",
        ".advertisement", ".ads", ".banner"
    ]
    for junk in container.select(', '.join(junk_tags)):
        junk.decompose()

    # 5. Remove ad-hoc promotional elements based on attributes
    ad_keywords = ["bitco-", "ad-", "sponsor", "promo", "subscribe", "newsletter"]
    for el in container.find_all(["div", "section", "aside"]):
        # Skip malformed elements that have no attrs (e.g., stray </>)
        if not getattr(el, "attrs", None):
            continue

        el_id = str(el.get("id", "")).lower()
        el_cls = "".join(str(c).lower() for c in el.get("class", []))
        el_style = str(el.get("style", "")).lower()

        if (
            any(k in el_id for k in ad_keywords)
            or any(k in el_cls for k in ad_keywords)
            or "display:none" in el_style
        ):
            el.decompose()

    # 6. Intelligent paragraph extraction
    # Recursive function to handle nested block elements correctly
    paragraphs = []
    block_tags = {
        'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote', 
        'div', 'section', 'article', 'li'
    }

    def process_node(node, buffer: List[str]):
        if isinstance(node, NavigableString):
            text = str(node).strip()
            if text:
                buffer.append(text)
            return

        if not hasattr(node, 'name'):
            return

        name = node.name.lower()

        # If block element, flush buffer to new paragraph
        if name in block_tags or name == 'br':
            if buffer:
                text = ' '.join(buffer).strip()
                if len(text) >= 15:
                    paragraphs.append(text)
                buffer.clear()
            
            # Recurse into children
            if name != 'br':
                for child in node.children:
                    process_node(child, buffer)
                
                # Flush again after block ends
                if buffer:
                    text = ' '.join(buffer).strip()
                    if len(text) >= 15:
                        paragraphs.append(text)
                    buffer.clear()
        else:
            # Inline elements (span, b, i, a) - keep adding to buffer
            for child in node.children:
                process_node(child, buffer)

    current_buffer = []
    for child in container.children:
        process_node(child, current_buffer)
    
    # Flush remaining buffer
    if current_buffer:
        text = ' '.join(current_buffer).strip()
        if len(text) >= 15:
            paragraphs.append(text)

    # 7. Post-processing and cleaning
    cleaned_paras = []
    skip_phrases = [
        "share this", "follow us", "click here", "read more", 
        "rights reserved", "tags:", "categories:"
    ]

    for p in paragraphs:
        p_lower = p.lower()
        
        # Skip junk phrases
        if any(s in p_lower for s in skip_phrases):
            continue
            
        # Clean whitespace and punctuation
        clean = re.sub(r'\s+', ' ', p).strip()
        clean = re.sub(r'\s*([.,;:!?])', r'\1', clean) # Remove space before punct
        clean = re.sub(r'(?<=[.,;:!?])(?=[^\s])', r' ', clean) # Ensure space after punct
        
        cleaned_paras.append(clean)

    data['content'] = '\n'.join(cleaned_paras) if cleaned_paras else ""
    return data

In [7]:
def fetch_data(row):
    """Fetch and extract article data safely, handling errors gracefully."""
    HTML_DIR = f"{NEWS_DIR}/html"
    
    try:
        # Direct Series access with proper error handling
        timestamp = row['timestamp']
        article_id = row['id']

        # Check if we got valid data
        if timestamp is None or article_id is None:
            print(f"[!] Missing required data: timestamp={timestamp}, id={article_id}")
            return pd.Series({})

        if pd.isnull(timestamp):
            raise ValueError("Timestamp is NaT")

        # Format timestamp to string (e.g., 20251028_060616), ignoring timezone
        ts_str = timestamp.strftime("%Y%m%d_%H%M%S")

        file_name = f"{int(article_id):06d}_{ts_str}.html"
        file_path = os.path.join(HTML_DIR, file_name)

        # Check if file exists before processing
        if not os.path.exists(file_path):
            print(f"[!] HTML file not found: {file_path}")
            return pd.Series({})

        # Call your extraction logic (extract_article works correctly!)
        result = extract_article(file_path)
        return pd.Series(result)

    except Exception as e:
        # Get ID safely for error reporting
        try:
            article_id = row['id']
        except:
            article_id = "Unknown"
        print(f"[!] Error processing ID {article_id}: {e}")
        # Return empty series to keep DataFrame structure aligned
        return pd.Series({})

In [8]:
def map_to_actionable_time(timestamp, timeframe_str):
    """
    Map timestamp to the NEXT actionable candle open time.
    Supports: m/min, h, d, w, M.
    """
    if pd.isnull(timestamp):
        return None
    
    if isinstance(timestamp, str):
        timestamp = pd.to_datetime(timestamp)
    
    # Extract amount and unit
    match = re.match(r"(\d+)?([a-zA-Z]+)(\d+)?", timeframe_str)
    if not match:
        raise ValueError(f"Invalid timeframe: {timeframe_str}")
    
    part1, unit_raw, part2 = match.groups()
    amount = int(part1 or part2 or "1")
    unit_raw = unit_raw.lower() # Normalize case

    # Logic Mapping
    try:
        if unit_raw in ['m', 'min', 'minute']:
            freq = f"{amount}min"
            # Floor to interval and shift +1 interval
            return timestamp.floor(freq) + pd.Timedelta(minutes=amount)

        elif unit_raw in ['h', 'hour']:
            freq = f"{amount}h"
            return timestamp.floor(freq) + pd.Timedelta(hours=amount)

        elif unit_raw in ['d', 'day']:
            freq = f"{amount}D"
            return timestamp.floor(freq) + pd.Timedelta(days=amount)

        elif unit_raw in ['w', 'week']:
            # Floor to Monday start of current week
            days_to_subtract = timestamp.weekday()
            start_of_week = (timestamp - pd.Timedelta(days=days_to_subtract)).normalize()
            # Shift to next week
            return start_of_week + pd.Timedelta(weeks=amount)

        elif unit_raw in ['m', 'mo', 'month']:
            # Floor to Start of Month
            start_of_month = timestamp.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
            # Shift to next month
            return start_of_month + pd.DateOffset(months=amount)

        else:
            raise ValueError(f"Unsupported unit: {unit_raw}")

    except Exception as e:
        print(f"[!] Error mapping time {timestamp}: {e}")
        return None

In [9]:
def format_merged_content(group):
    """
    Helper to merge multiple articles into a single structured string.
    """
    merged_text = ""
    # Sort by timestamp inside the group to keep chronological order of news
    group = group.sort_values('timestamp')
    
    for _, row in group.iterrows():
        title = row.get('title', 'No Title')
        content = row.get('content', '')
        # tags = row.get('tags', [])
        # author = row.get('author', 'Unknown Author')
        # date_str = row.get('date', '')
        # category = row.get('category', 'General')
        
        # Structure the text for LLM/Model input
        merged_text += f"# [Article ID: {row['id']}] {title}\n"
        # merged_text += f"**Meta:** {author} - {category} - {date_str} | **Tags:** {tags}\n"
        merged_text += f"{content}\n"
        merged_text += "---\n"
        
    return merged_text.strip()

In [10]:
def process_and_merge_news(df: pd.DataFrame, timeframe: str = '4h') -> pd.DataFrame:
    """
    Groups news by actionable timeframe and merges content.
    Returns a clean DataFrame indexed by actionable_time.
    """
    print(f"Grouping news by timeframe: {timeframe}...")
    
    # 1. Create Actionable Time Column
    # Ensure timestamp is datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    df['actionable_time'] = df['timestamp'].apply(
        lambda x: map_to_actionable_time(x, timeframe)
    )
    
    # Remove rows where time mapping failed
    df_clean = df.dropna(subset=['actionable_time'])

    # 2. Group and Aggregate
    # We aggregate content using the helper, and count the number of articles
    grouped = df_clean.groupby('actionable_time').apply(
        lambda x: pd.Series({
            'merged_content': format_merged_content(x),
            'article_count': len(x),
            'original_ids': x['id'].tolist()
        })
    )
    
    # 3. Sort index (Time)
    grouped = grouped.sort_index()
    
    return grouped

In [11]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from huggingface_hub import InferenceClient


def mask_key(key: str, head: int = 5, tail: int = 4) -> str:
    """Mask the middle portion of a key using replace as requested."""
    if not key or len(key) <= head + tail:
        return "****"
    return key.replace(key[head:-tail], "****")


def init_cryptobert_classifier(
    hf_key: str,
    model_name: str,
    batch_size_gpu: int = 32,
    batch_size_cpu: int = 8,
    device: int = None,
):
    """Initialize CryptoBERT tokenizer and classifier with configurable batch sizes.
    Returns (tokenizer, classify_batch_fn).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    resolved_device = device if device is not None else (0 if torch.cuda.is_available() else -1)

    if resolved_device >= 0:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            torch_dtype=(
                torch.bfloat16
                if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
                else torch.float16
            ),
        )
        classifier = pipeline(
            "text-classification",
            model=model,
            tokenizer=tokenizer,
            device=resolved_device,
            top_k=None,
        )
        default_batch_size = batch_size_gpu

        def _classify_batch(texts, batch_size=None):
            effective_batch = batch_size or default_batch_size
            outs = classifier(texts, batch_size=effective_batch)
            res = []
            for out in outs:
                score_map = {o["label"]: o["score"] for o in out}
                res.append([
                    score_map.get("Bullish", 0.0),
                    score_map.get("Neutral", 0.0),
                    score_map.get("Bearish", 0.0),
                ])
            return res
    else:
        client = InferenceClient(
            provider="auto",
            api_key=hf_key,
        )
        default_batch_size = batch_size_cpu

        def _classify_batch(texts, batch_size=None):
            res = []
            for text in texts:
                result = client.text_classification(text=text, model=model_name)
                score_map = {}
                for x in result:
                    label = getattr(x, "label", None) or x["label"]
                    score = getattr(x, "score", None) or x["score"]
                    score_map[label] = score
                res.append([
                    score_map.get("Bullish", 0.0),
                    score_map.get("Neutral", 0.0),
                    score_map.get("Bearish", 0.0),
                ])
            return res

    return tokenizer, _classify_batch


def chunk_by_tokens(
    text: str,
    tokenizer,
    chunk_size: int = 256,
    overlap: int = 48,
    max_chunks: int = 10,
):
    """Split text into token-based chunks with overlap."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(tokens) and len(chunks) < max_chunks:
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start += step

    return chunks


def cryptobert_sentiment_long_article(
    title: str,
    content: str,
    tokenizer,
    classify_batch,
    chunk_size: int = 256,
    overlap: int = 48,
    max_chunks: int = 10,
    topk: int = 3,
    head_char_limit: int = 1000,
):
    """Compute sentiment for a long article with head-focused and global aggregation."""
    title = title or ""
    content = content or ""

    full_text = f"{title}\n\n{content}"

    # 1) HEAD: title + first ~1000 characters
    head_text = f"{title}\n\n{content[:head_char_limit]}"
    head_chunks = chunk_by_tokens(
        head_text,
        tokenizer,
        chunk_size=chunk_size,
        overlap=overlap,
        max_chunks=max_chunks,
    )

    head_scores = (
        np.array(classify_batch(head_chunks))
        if head_chunks
        else np.zeros((1, 3))
    )
    head_mean = head_scores.mean(axis=0)

    # 2) Global view: entire article
    all_chunks = chunk_by_tokens(
        full_text,
        tokenizer,
        chunk_size=chunk_size,
        overlap=overlap,
        max_chunks=max_chunks,
    )

    all_scores = (
        np.array(classify_batch(all_chunks))
        if all_chunks
        else np.zeros((1, 3))
    )

    global_mean = all_scores.mean(axis=0)
    global_max = all_scores.max(axis=0)

    k = min(topk, all_scores.shape[0])
    if k > 0:
        idx_bull = np.argsort(-all_scores[:, 0])[:k]
        idx_bear = np.argsort(-all_scores[:, 2])[:k]
        topk_bull_mean = all_scores[idx_bull, 0].mean()
        topk_bear_mean = all_scores[idx_bear, 2].mean()
    else:
        topk_bull_mean = 0.0
        topk_bear_mean = 0.0

    return {
        "head_p_bull": float(head_mean[0]),
        "head_p_neu":  float(head_mean[1]),
        "head_p_bear": float(head_mean[2]),

        "mean_p_bull": float(global_mean[0]),
        "mean_p_neu":  float(global_mean[1]),
        "mean_p_bear": float(global_mean[2]),

        "max_p_bull": float(global_max[0]),
        "max_p_neu":  float(global_max[1]),
        "max_p_bear": float(global_max[2]),

        "topk_mean_p_bull": float(topk_bull_mean),
        "topk_mean_p_bear": float(topk_bear_mean),

        "head_sent_net": float(head_mean[0] - head_mean[2]),
        "global_sent_net": float(global_mean[0] - global_mean[2]),
    }


## Merge

### Load

In [12]:
def load_data(label, path):
    """Load CSV file and return DataFrame with info display"""
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        # Handle different timestamp column names
        timestamp_col = 'timestamp'  # default
        if timestamp_col not in df.columns:
            # Try common alternatives
            for col in ['datetime', 'time', 'date']:
                if col in df.columns:
                    timestamp_col = col
                    break
        
        if timestamp_col and timestamp_col in df.columns:
            print(f"{label}: {len(df)} records ({df[timestamp_col].min()} → {df[timestamp_col].max()})")
        else:
            print(f"{label}: {len(df)} records (no timestamp column found)")
        
        return df
    else:
        print(f"✗ {label}: File not found")
        return None

# Data sources configuration
DATA_SOURCES = [
    ("OHLCV", COINS, get_ohlcv_file),
    ("Market Cap", None, get_marketcap_file),
    ("Network Activity", COINS, get_networkactivity_file),
    ("Mining", None, get_mining_file),
    ("Profit & Value", COINS, get_profitandvalue_file),
    ("Holder Behavior", ["BTC"], get_holderbehavior_file),
    ("News", None, get_news_file),
    ("Sentiment Index", None, get_sentimentindex_file),
]

# Load all data sources into DataFrames
dataframes = {}

for source_name, coins, get_file_func in DATA_SOURCES:
    if coins:
        # Multiple coins for this source type
        for coin in coins:
            label = f"{coin} {source_name}"
            df = load_data(label, get_file_func(coin))
            if df is not None:
                # Store with clean key name
                key = f"{coin.lower()}_{source_name.lower().replace(' ', '_')}"
                dataframes[key] = df
    else:
        # Single file for this source type
        df = load_data(source_name, get_file_func())
        if df is not None:
            # Store with clean key name
            key = source_name.lower().replace(' ', '_')
            dataframes[key] = df
    print("-"*10)

print(f"\n✓ Loaded {len(dataframes)} DataFrames:")
print(list(dataframes.keys()))

BTC OHLCV: 18117 records (2017-08-17 04:00:00 → 2025-11-25 04:00:00)
ETH OHLCV: 18117 records (2017-08-17 04:00:00 → 2025-11-25 04:00:00)
----------
Market Cap: 11189 records (2013-04-27 23:59:59.999000+00:00 → 2025-11-10 07:59:59.999000+00:00)
----------
BTC Network Activity: 6155 records (2009-01-03 00:00:00+00:00 → 2025-11-09 00:00:00+00:00)
ETH Network Activity: 3756 records (2015-07-30 00:00:00+00:00 → 2025-11-09 00:00:00+00:00)
----------
Mining: 3075 records (2009-01-03 → 2025-11-09)
----------
BTC Profit & Value: 6156 records (2009-01-03 00:00:00+00:00 → 2025-11-10 00:00:00+00:00)
ETH Profit & Value: 3757 records (2015-07-30 00:00:00+00:00 → 2025-11-10 00:00:00+00:00)
----------
BTC Holder Behavior: 6156 records (2009-01-03 00:00:00+00:00 → 2025-11-10 00:00:00+00:00)
----------
News: 13400 records (2012-02-28T06:06:16+00:00 → 2025-12-02T04:29:31.546517+00:00)
----------
Sentiment Index: 2841 records (2018-02-01 → 2025-11-15)
----------

✓ Loaded 11 DataFrames:
['btc_ohlcv', 'et

#### OHLCV

##### BTC

In [13]:
dataframes["btc_ohlcv"].head(5)

,timestamp,open,high,low,close,volume,quote_volume,trades
0,2017-08-17 04:00:00,4261.48,4349.99,4261.32,4349.99,82.088865,3.531943e+05,334
1,2017-08-17 08:00:00,4333.32,4485.39,4333.32,4427.30,63.619882,2.825012e+05,248
2,2017-08-17 12:00:00,4436.06,4485.39,4333.42,4352.34,174.562001,7.742388e+05,858
3,2017-08-17 16:00:00,4352.33,4354.84,4200.74,4325.23,225.109716,9.652911e+05,986
4,2017-08-17 20:00:00,4307.56,4369.69,4258.56,4285.08,249.769913,1.079545e+06,1001


In [14]:
dataframes["btc_ohlcv"] = convert_timestamp(dataframes["btc_ohlcv"], "timestamp")
dataframes["btc_ohlcv"].head(5)

,timestamp,open,high,low,close,volume,quote_volume,trades
0,2017-08-17 04:00:00+00:00,4261.48,4349.99,4261.32,4349.99,82.088865,3.531943e+05,334
1,2017-08-17 08:00:00+00:00,4333.32,4485.39,4333.32,4427.30,63.619882,2.825012e+05,248
2,2017-08-17 12:00:00+00:00,4436.06,4485.39,4333.42,4352.34,174.562001,7.742388e+05,858
3,2017-08-17 16:00:00+00:00,4352.33,4354.84,4200.74,4325.23,225.109716,9.652911e+05,986
4,2017-08-17 20:00:00+00:00,4307.56,4369.69,4258.56,4285.08,249.769913,1.079545e+06,1001


##### ETH

In [15]:
dataframes["eth_ohlcv"].head(5)

,timestamp,open,high,low,close,volume,quote_volume,trades
0,2017-08-17 04:00:00,301.13,307.96,298.00,307.96,1561.95305,473487.665119,711
1,2017-08-17 08:00:00,307.95,312.00,307.00,308.95,1177.71088,364545.316402,775
2,2017-08-17 12:00:00,308.95,310.51,303.56,307.06,1882.05267,578644.931890,1140
3,2017-08-17 16:00:00,307.74,312.18,298.21,301.60,1208.05192,370209.051467,957
4,2017-08-17 20:00:00,301.60,310.85,299.01,302.00,1200.94182,367768.335479,939


In [16]:
dataframes["eth_ohlcv"] = convert_timestamp(dataframes["eth_ohlcv"], "timestamp")
dataframes["eth_ohlcv"].head(5)

,timestamp,open,high,low,close,volume,quote_volume,trades
0,2017-08-17 04:00:00+00:00,301.13,307.96,298.00,307.96,1561.95305,473487.665119,711
1,2017-08-17 08:00:00+00:00,307.95,312.00,307.00,308.95,1177.71088,364545.316402,775
2,2017-08-17 12:00:00+00:00,308.95,310.51,303.56,307.06,1882.05267,578644.931890,1140
3,2017-08-17 16:00:00+00:00,307.74,312.18,298.21,301.60,1208.05192,370209.051467,957
4,2017-08-17 20:00:00+00:00,301.60,310.85,299.01,302.00,1200.94182,367768.335479,939


#### Marketcap

In [17]:
dataframes["market_cap"].head(5)

,timestamp,BTC_market_cap,ETH_market_cap,USDT_market_cap,USDC_market_cap
0,2013-04-27 23:59:59.999000+00:00,1.515588e+09,0.0,0.0,0.0
1,2013-04-28 07:59:59.999000+00:00,1.291704e+09,0.0,0.0,0.0
2,2013-04-28 19:59:59.999000+00:00,1.469491e+09,0.0,0.0,0.0
3,2013-04-29 07:59:59.999000+00:00,1.503054e+09,0.0,0.0,0.0
4,2013-04-29 19:59:59.999000+00:00,1.569978e+09,0.0,0.0,0.0


In [18]:
dataframes["market_cap"] = convert_timestamp(dataframes["market_cap"], "timestamp", round_to="s")
dataframes["market_cap"].head(5)

,timestamp,BTC_market_cap,ETH_market_cap,USDT_market_cap,USDC_market_cap
0,2013-04-27 23:59:59+00:00,1.515588e+09,0.0,0.0,0.0
1,2013-04-28 07:59:59+00:00,1.291704e+09,0.0,0.0,0.0
2,2013-04-28 19:59:59+00:00,1.469491e+09,0.0,0.0,0.0
3,2013-04-29 07:59:59+00:00,1.503054e+09,0.0,0.0,0.0
4,2013-04-29 19:59:59+00:00,1.569978e+09,0.0,0.0,0.0


#### Network activity

In [19]:
dataframes["btc_network_activity"].head(5)

,timestamp,active_addresses,tx_count
0,2009-01-03 00:00:00+00:00,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0


In [20]:
dataframes['eth_network_activity'].head(5)

,timestamp,active_addresses,tx_count
0,2015-07-30 00:00:00+00:00,9206.0,0.0
1,2015-07-31 00:00:00+00:00,424.0,0.0
2,2015-08-01 00:00:00+00:00,413.0,0.0
3,2015-08-02 00:00:00+00:00,432.0,0.0
4,2015-08-03 00:00:00+00:00,444.0,0.0


#### Mining

In [21]:
dataframes["mining"].head(5)

,timestamp,mining_difficulty,hash_rate_ths,miner_revenue_usd
0,2009-01-03,1.0,4.971027e-08,0.0
1,2009-01-07,0.0,0.000000e+00,0.0
2,2009-01-11,1.0,5.269289e-06,0.0
3,2009-01-15,1.0,6.313204e-06,0.0
4,2009-01-17,1.0,6.313204e-06,0.0


In [22]:
dataframes["mining"] = convert_timestamp(dataframes["mining"], "timestamp")
dataframes["mining"].head(5)

,timestamp,mining_difficulty,hash_rate_ths,miner_revenue_usd
0,2009-01-03 00:00:00+00:00,1.0,4.971027e-08,0.0
1,2009-01-07 00:00:00+00:00,0.0,0.000000e+00,0.0
2,2009-01-11 00:00:00+00:00,1.0,5.269289e-06,0.0
3,2009-01-15 00:00:00+00:00,1.0,6.313204e-06,0.0
4,2009-01-17 00:00:00+00:00,1.0,6.313204e-06,0.0


#### Profit and Value

##### BTC

In [23]:
dataframes["btc_profit_&_value"].head(5)

,timestamp,mvrv_ratio,exchange_inflow_native,exchange_inflow_usd,exchange_outflow_native,exchange_outflow_usd,exchange_supply_native,exchange_supply_usd
0,2009-01-03 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### ETH

In [24]:
dataframes["eth_profit_&_value"].head(5)

,timestamp,mvrv_ratio,exchange_inflow_native,exchange_inflow_usd,exchange_outflow_native,exchange_outflow_usd,exchange_supply_native,exchange_supply_usd
0,2015-07-30 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-07-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-08-01 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-08-02 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-08-03 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Holder behavior

In [25]:
dataframes["btc_holder_behavior"].head(5)

,timestamp,total_supply,mvrv_ratio,exchange_inflow_native,exchange_inflow_usd,exchange_outflow_native,exchange_outflow_usd,exchange_supply_native,exchange_supply_usd
0,2009-01-03 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-01-04 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-01-05 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-01-06 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-01-07 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### News

In [26]:
pd.concat([
    dataframes['news'].head(5),
    dataframes['news'].tail(5)
])

,id,datetime,url,status
0,1,2012-02-28T06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1
1,2,2012-02-28T11:34:40+00:00,https://bitcoinmagazine.com/culture/bitcoin-ad...,1
2,3,2012-02-28T22:46:22+00:00,https://bitcoinmagazine.com/technical/bitcoin-...,1
3,4,2012-02-28T23:24:16+00:00,https://bitcoinmagazine.com/technical/client-s...,1
4,5,2012-02-29T03:24:31+00:00,https://bitcoinmagazine.com/culture/traditiona...,1
13395,13396,2025-12-02T04:29:31.546506+00:00,https://bitcoinmagazine.com/news/crypto-house-...,1
13396,13397,2025-12-02T04:29:31.546509+00:00,https://bitcoinmagazine.com/markets/bitcoins-n...,1
13397,13398,2025-12-02T04:29:31.546512+00:00,https://bitcoinmagazine.com/markets/why-bitcoi...,1
13398,13399,2025-12-02T04:29:31.546514+00:00,https://bitcoinmagazine.com/business/strategy-...,1
13399,13400,2025-12-02T04:29:31.546517+00:00,https://bitcoinmagazine.com/news/bitcoin-price...,1


In [27]:
dataframes['news'] = dataframes['news'].rename(columns={"datetime": "timestamp"})
dataframes['news'] = convert_timestamp(dataframes['news'], 'timestamp', round_to="s")
pd.concat([
    dataframes['news'].head(5),
    dataframes['news'].tail(5)
])

,id,timestamp,url,status
0,1,2012-02-28 06:06:16+00:00,https://bitcoinmagazine.com/business/the-waste...,1
1,2,2012-02-28 11:34:40+00:00,https://bitcoinmagazine.com/culture/bitcoin-ad...,1
2,3,2012-02-28 22:46:22+00:00,https://bitcoinmagazine.com/technical/bitcoin-...,1
3,4,2012-02-28 23:24:16+00:00,https://bitcoinmagazine.com/technical/client-s...,1
4,5,2012-02-29 03:24:31+00:00,https://bitcoinmagazine.com/culture/traditiona...,1
13395,13396,2025-12-02 04:29:31+00:00,https://bitcoinmagazine.com/news/crypto-house-...,1
13396,13397,2025-12-02 04:29:31+00:00,https://bitcoinmagazine.com/markets/bitcoins-n...,1
13397,13398,2025-12-02 04:29:31+00:00,https://bitcoinmagazine.com/markets/why-bitcoi...,1
13398,13399,2025-12-02 04:29:31+00:00,https://bitcoinmagazine.com/business/strategy-...,1
13399,13400,2025-12-02 04:29:31+00:00,https://bitcoinmagazine.com/news/bitcoin-price...,1


In [28]:
### Crawl News
# dataframes['news'] = dataframes['news'].progress_apply(fetch_data, axis=1).combine_first(dataframes['news'])
dataframes['news'] = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_crawl.csv", sep='\t')

In [29]:
pd.concat([
    dataframes['news'].head(5),
    dataframes['news'].tail(5)
])

,author,content,date,id,status,tags,timestamp,title,url
0,Vitalik Buterin,One of the main arguments in favor of fiat cur...,2012-02-28T06:06:16-05:00,1,1,"['energy consumption', 'energy waste', 'Fees',...",2012-02-28 06:06:16+00:00,The Wasted Electricity Objection To Bitcoin,https://bitcoinmagazine.com/business/the-waste...
1,Vitalik Buterin,If Bitcoin is to achieve mainstream success it...,2012-02-28T11:34:40-05:00,2,1,"['Cards', 'Internet', 'People']",2012-02-28 11:34:40+00:00,Bitcoin Adoption Opportunity: Teenagers,https://bitcoinmagazine.com/culture/bitcoin-ad...
2,Vitalik Buterin,"In a managed online wallet, everything is cont...",2012-02-28T22:46:22-05:00,3,1,"['Hacks', 'Mtgox', 'Wallets']",2012-02-28 22:46:22+00:00,Bitcoin Exchange Wallets,https://bitcoinmagazine.com/technical/bitcoin-...
3,Vitalik Buterin,Client side browser wallets almost exactly res...,2012-02-28T23:24:16-05:00,4,1,[],2012-02-28 23:24:16+00:00,Client Side Secured Browser Wallets,https://bitcoinmagazine.com/technical/client-s...
4,Mihai Alisie,Traditional Bitcoin clients are ordinary progr...,2012-02-29T03:24:31-05:00,5,1,['Wallets'],2012-02-29 03:24:31+00:00,Traditional Bitcoin Client/Wallet,https://bitcoinmagazine.com/culture/traditiona...
13386,Micah Zimmerman,SoFi Technologies (NASDAQ: SOFI) has become th...,2025-11-11T09:52:03-05:00,13387,1,"['Banking', 'Bitcoin', 'Bitcoin Trading', 'Ret...",2025-11-13 18:38:28+00:00,SoFi Enters The Bitcoin Era As First U.S. Bank...,https://bitcoinmagazine.com/news/sofi-makes-hi...
13387,Matt Crosby,Bitcoin price long-term trajectory has been ex...,2025-11-11T09:29:33-05:00,13388,1,"['Bitcoin', 'Bitcoin Magazine Pro', 'bitcoin p...",2025-11-13 18:38:33+00:00,Bitcoin Price Outlook For Reaching $1 Million ...,https://bitcoinmagazine.com/markets/bitcoin-pr...
13388,Juan Galt,"Lendasat, a Bitcoin-native peer-to-peer lendin...",2025-11-14T04:00:00-05:00,13389,1,[],2025-11-14 09:57:17+00:00,What If You Could Swap Bitcoin For Stablecoins...,https://bitcoinmagazine.com/business/lendaswap...
13389,Micah Zimmerman,"Bitcoin price fell sharply today, sliding from...",2025-11-13T15:32:13-05:00,13390,1,[],2025-11-14 09:57:17+00:00,"Bitcoin Price Crashes Below $98,000 To 6-Month...",https://bitcoinmagazine.com/markets/bitcoin-pr...
13390,Micah Zimmerman,"Bitfarms, one of North America’s largest Bitco...",2025-11-13T11:29:20-05:00,13391,1,"['Artificial Intelligence', 'Bitcoin miniing',...",2025-11-14 09:57:17+00:00,"Bitfarms (BITF) To Exit Bitcoin Mining, Pivot ...",https://bitcoinmagazine.com/news/bitfarms-to-e...


In [30]:
candidates = [",", ";", "|", "\t", "~", "^"]

df_str = dataframes['news'].astype(str)

safe_seps = []
bad_seps = []

for sep in candidates:
    exists = df_str.apply(lambda col: col.str.contains(sep, regex=False)).any().any()
    if exists:
        bad_seps.append(sep)
    else:
        safe_seps.append(sep)

print("✅:", safe_seps)
print("❌:", bad_seps)

✅: ['\t']
❌: [',', ';', '|', '~', '^']


In [31]:
dataframes['news'].to_csv(f"{NEWS_DIR}/bitcoinmagazinenews_crawl.csv", sep='\t', index=False)
df_news = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_crawl.csv", sep='\t')
pd.concat([
    df_news.head(1),
    df_news.tail(1)
])

,author,content,date,id,status,tags,timestamp,title,url
0,Vitalik Buterin,One of the main arguments in favor of fiat cur...,2012-02-28T06:06:16-05:00,1,1,"['energy consumption', 'energy waste', 'Fees',...",2012-02-28 06:06:16+00:00,The Wasted Electricity Objection To Bitcoin,https://bitcoinmagazine.com/business/the-waste...
13390,Micah Zimmerman,"Bitfarms, one of North America’s largest Bitco...",2025-11-13T11:29:20-05:00,13391,1,"['Artificial Intelligence', 'Bitcoin miniing',...",2025-11-14 09:57:17+00:00,"Bitfarms (BITF) To Exit Bitcoin Mining, Pivot ...",https://bitcoinmagazine.com/news/bitfarms-to-e...


In [32]:
nan_rows = df_news[df_news["content"].isna() | df_news["title"].isna()]
len(nan_rows), str(round(len(nan_rows)/len(df_news)*100, 2)) + "%"

(85, '0.63%')

In [33]:
df_news = df_news.dropna(subset=["content", "title"])
new_nan_rows = df_news[df_news["content"].isna() | df_news["title"].isna()]
len(new_nan_rows), str(round(len(new_nan_rows)/len(df_news)*100, 2)) + "%"

(0, '0.0%')

In [34]:
# SENTIMENT_CONFIG = {
#     "model_name": "ElKulako/cryptobert",
#     "chunk_size": 256,
#     "overlap": 48,
#     "max_chunks": 16,
#     "topk": 3,
#     "head_char_limit": 1000,
#     "batch_size_gpu": 256,
#     "batch_size_cpu": 8,
#     "device": None,  # override to force CPU/GPU; None keeps auto selection
# }

# if hf_key is None:
#     print("Warning: hf_key not found in environment variables, please set it in the .env file.")
# else:
#     print("✅ Loaded hf_key:", mask_key(hf_key))

#     tokenizer, classify_batch = init_cryptobert_classifier(
#         hf_key=hf_key,
#         model_name=SENTIMENT_CONFIG["model_name"],
#         batch_size_gpu=SENTIMENT_CONFIG["batch_size_gpu"],
#         batch_size_cpu=SENTIMENT_CONFIG["batch_size_cpu"],
#         device=SENTIMENT_CONFIG["device"],
#     )

#     scores = [
#         cryptobert_sentiment_long_article(
#             title=title,
#             content=content,
#             tokenizer=tokenizer,
#             classify_batch=classify_batch,
#             chunk_size=SENTIMENT_CONFIG["chunk_size"],
#             overlap=SENTIMENT_CONFIG["overlap"],
#             max_chunks=SENTIMENT_CONFIG["max_chunks"],
#             topk=SENTIMENT_CONFIG["topk"],
#             head_char_limit=SENTIMENT_CONFIG["head_char_limit"],
#         )
#         for title, content in tqdm(
#             zip(df_news["title"], df_news["content"]),
#             total=len(df_news)
#         )
#     ]

#     score_df = pd.DataFrame(scores)

#     df_news = pd.concat([df_news.reset_index(drop=True), score_df], axis=1)

# df_news.head(5)
# df_news.to_csv(f"{NEWS_DIR}/bitcoinmagazinenews_extract.csv", sep='\t', index=False)

In [37]:
df_news = pd.read_csv(f"{NEWS_DIR}/bitcoinmagazinenews_extract.csv", sep='\t')
pd.concat([
    df_news.head(1),
    df_news.tail(1)
])

,author,content,date,id,status,tags,timestamp,title,url,head_p_bull,...,mean_p_bull,mean_p_neu,mean_p_bear,max_p_bull,max_p_neu,max_p_bear,topk_mean_p_bull,topk_mean_p_bear,head_sent_net,global_sent_net
0,Vitalik Buterin,One of the main arguments in favor of fiat cur...,2012-02-28T06:06:16-05:00,1,1,"['energy consumption', 'energy waste', 'Fees',...",2012-02-28 06:06:16+00:00,The Wasted Electricity Objection To Bitcoin,https://bitcoinmagazine.com/business/the-waste...,0.633942,...,0.350466,0.288996,0.360538,0.691526,0.595159,0.806509,0.512493,0.610582,0.500018,-0.010072
13305,Micah Zimmerman,"Bitfarms, one of North America’s largest Bitco...",2025-11-13T11:29:20-05:00,13391,1,"['Artificial Intelligence', 'Bitcoin miniing',...",2025-11-14 09:57:17+00:00,"Bitfarms (BITF) To Exit Bitcoin Mining, Pivot ...",https://bitcoinmagazine.com/news/bitfarms-to-e...,0.335204,...,0.337249,0.646748,0.016003,0.393755,0.673965,0.036722,0.337249,0.016003,0.308648,0.321247


In [ ]:
dataframes['news'] = process_and_merge_news(dataframes['news'], timeframe='4h')
dataframes['news']

In [ ]:
# print sample merged content
sample_times = dataframes['news'].index[:2]
for t in sample_times:
    print(dataframes['news'].loc[t, 'merged_content'])
    print("\n\n")

#### Sentiment index

In [ ]:
dataframes['sentiment_index'].head(5)

In [ ]:
dataframes['sentiment_index'] = convert_timestamp(dataframes['sentiment_index'], 'timestamp')
dataframes['sentiment_index'].head(5)

In [ ]:
raise Exception("STOPPPPP")

In [ ]:
df